# K-Means From Scratch — Step-by-Step Walkthrough
Each stage is printed and plotted so you can watch the algorithm converge.

## Task 1 — Load and visualize the dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.io import loadmat

data = loadmat('ex7data2.mat')
X = data['X']

print(f'Shape: {X.shape}  ({X.shape[0]} points, {X.shape[1]} features)')
print('First 5 rows:')
print(X[:5])

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], s=15, color='steelblue', alpha=0.6)
plt.title('Raw data — no labels yet')
plt.xlabel('x1'); plt.ylabel('x2')
plt.tight_layout()
plt.show()


## Task 2 — Finding closest centroids

In [ ]:
def find_closest_centroids(X, centroids):
    # X: (m,1,n) broadcast against centroids: (1,K,n) -> distances: (m,K)
    distances = np.sum((X[:, np.newaxis, :] - centroids[np.newaxis, :, :]) ** 2, axis=2)
    return np.argmin(distances, axis=1)


## Task 3 — Computing centroids

In [ ]:
def compute_centroids(X, idx, K, rng=None):
    m, n = X.shape
    rng = np.random.default_rng() if rng is None else rng
    new_centroids = np.zeros((K, n), dtype=float)
    for k in range(K):
        pts = X[idx == k]
        new_centroids[k] = X[rng.integers(0, m)] if pts.shape[0] == 0 else pts.mean(axis=0)
    return new_centroids


## Task 4 — Running K-Means with step-by-step output

In [ ]:
COLORS = ['#e74c3c', '#2ecc71', '#3498db']  # one color per cluster

def plot_state(X, idx, centroids, title):
    plt.figure(figsize=(6, 5))
    K = centroids.shape[0]
    for k in range(K):
        mask = idx == k
        plt.scatter(X[mask, 0], X[mask, 1], s=15, color=COLORS[k], alpha=0.5, label=f'Cluster {k}')
    plt.scatter(
        centroids[:, 0], centroids[:, 1],
        marker='*', s=350, c=COLORS[:K],
        edgecolors='black', linewidths=0.8, zorder=5, label='Centroids'
    )
    plt.title(title)
    plt.xlabel('x1'); plt.ylabel('x2')
    plt.legend(loc='upper right', fontsize=8)
    plt.tight_layout()
    plt.show()


def run_k_means(X, init_centroids, max_iters=10, rng=None):
    K = init_centroids.shape[0]
    centroids = init_centroids.copy().astype(float)
    history = [centroids.copy()]
    idx = None

    print('=' * 50)
    print(f'STAGE 0 -- Randomly picked {K} initial centroids')
    print('=' * 50)
    for k, c in enumerate(centroids):
        print(f'  Centroid {k}: ({c[0]:.3f}, {c[1]:.3f})')

    plt.figure(figsize=(6, 5))
    plt.scatter(X[:, 0], X[:, 1], s=15, color='steelblue', alpha=0.4, label='Data')
    plt.scatter(
        centroids[:, 0], centroids[:, 1],
        marker='*', s=350, c=COLORS[:K],
        edgecolors='black', linewidths=0.8, zorder=5, label='Initial centroids'
    )
    plt.title('Stage 0 -- Random initial centroids')
    plt.xlabel('x1'); plt.ylabel('x2')
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    for it in range(max_iters):
        print()
        print('=' * 50)
        print(f'ITERATION {it + 1} / {max_iters}')
        print('=' * 50)

        print(f'\n  >> STEP A: Assigning each point to its nearest centroid...')
        idx = find_closest_centroids(X, centroids)
        counts = {k: int((idx == k).sum()) for k in range(K)}
        for k, n in counts.items():
            print(f'     Cluster {k}: {n} points assigned')
        plot_state(X, idx, centroids, f'Iter {it+1} Step A — points assigned to centroids')

        print(f'\n  >> STEP B: Recomputing centroids as cluster means...')
        old_centroids = centroids.copy()
        centroids = compute_centroids(X, idx, K, rng=rng)
        history.append(centroids.copy())
        for k in range(K):
            shift = np.linalg.norm(centroids[k] - old_centroids[k])
            print(f'     Centroid {k}: ({centroids[k][0]:.3f}, {centroids[k][1]:.3f})  [moved {shift:.4f}]')
        plot_state(X, idx, centroids, f'Iter {it+1} Step B — centroids recomputed')

        total_shift = np.sum(np.linalg.norm(centroids - old_centroids, axis=1))
        if total_shift < 1e-6:
            print(f'\n  Converged after {it+1} iterations (total shift < 1e-6).')
            break

    print()
    print('=' * 50)
    print('DONE -- Final cluster assignments')
    print('=' * 50)
    return centroids, idx, history


## Task 5 — Initializing centroids

In [ ]:
def init_centroids(X, K, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    idx = rng.choice(X.shape[0], size=K, replace=False)  # K distinct random points
    return X[idx].astype(float)

rng0 = np.random.default_rng(42)
c0 = init_centroids(X, 3, rng0)
print('Initial centroids (K=3):')
print(c0)

final_centroids, idx, hist = run_k_means(X, c0, max_iters=10, rng=np.random.default_rng(42))


## Task 6 — Multiple initializations and best run

In [ ]:
def withinss(X, centroids, idx):
    diffs = X - centroids[idx]           # each point minus its assigned centroid
    return float(np.sum(diffs * diffs))  # total within-cluster sum of squares

print('Running 5 random seeds, keeping the best...\n')
best = {'score': np.inf}
for seed in range(5):
    rng = np.random.default_rng(seed)
    c0 = init_centroids(X, 3, rng)
    cF, idxF, hist = run_k_means(X, c0, max_iters=10, rng=np.random.default_rng(seed))
    s = withinss(X, cF, idxF)
    print(f'Seed {seed}: inertia = {s:.2f}')
    if s < best['score']:
        best = {'score': s, 'centroids': cF, 'idx': idxF}

print(f'\nBest inertia: {best["score"]:.4f}')
plot_state(X, best['idx'], best['centroids'], f'Best run -- inertia = {best["score"]:.2f}')

rep = pd.DataFrame({'x1': X[:, 0], 'x2': X[:, 1], 'cluster': best['idx']})
rep.to_csv('kmeans_assignments.csv', index=False)
print('Wrote kmeans_assignments.csv')
